# Experiment 06 — API Testing Agent using MCP Protocol

Developer Tools Domain | Google Colab | Optional Groq `openai/gpt-oss-20b`

This notebook follows the supplied MCP experiment structure: OpenAPI retrieval, MCP server/client over stdio, dynamic tool discovery, planner/supervisor, policy validation, API tests, concurrency, dependency handling, tracing, recovery, performance benchmarking, and JSON reporting. Actual network execution is restricted to allowlisted read-only GET endpoints on the public Swagger Petstore demo.

**Aim:** Implement an MCP-based API Testing and Test-Case Generation Agent that inspects a real OpenAPI specification, generates test cases, executes approved requests, validates responses, and records performance and execution traces.


In [ ]:
%pip -q install "mcp>=1.9,<2" requests openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 4.7 MB/s eta 0:00:00


## Run the complete implementation
Run this cell after installing dependencies. Add `GROQ_API_KEY` to Colab Secrets to enable Groq planning. Without a key, the deterministic planner is used.


In [ ]:
import os, sys, json, time, uuid, asyncio, logging, ast, subprocess
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean
from urllib.parse import urlparse, urljoin
import requests
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

ROOT=Path('/content/api_testing_agent'); PROJECT=ROOT/'sample_developer_project'; LOG_DIR=Path('/content/mcp_logs')
ROOT.mkdir(parents=True,exist_ok=True); PROJECT.mkdir(parents=True,exist_ok=True); LOG_DIR.mkdir(parents=True,exist_ok=True)
SPEC_FILE=ROOT/'petstore_openapi.json'; SERVER_FILE=ROOT/'mcp_server.py'
TRACE_FILE=LOG_DIR/'mcp_traces.jsonl'; LOG_FILE=LOG_DIR/'execution.log'
REPORT_FILE=LOG_DIR/'developer_api_automation_report.json'; SUMMARY_FILE=LOG_DIR/'experiment_06_execution_summary.json'
for p in [TRACE_FILE]:
    if p.exists(): p.unlink()
logging.basicConfig(level=logging.INFO,format='%(asctime)s | %(levelname)s | %(message)s',handlers=[logging.FileHandler(LOG_FILE),logging.StreamHandler()])
logger=logging.getLogger('developer-api-agent')
OPENAPI_URL='https://petstore3.swagger.io/api/v3/openapi.json'; API_BASE='https://petstore3.swagger.io'
ALLOWED_HOSTS={'petstore3.swagger.io'}
ALLOWED_ENDPOINTS={'/api/v3/pet/findByStatus','/api/v3/pet/1','/api/v3/store/inventory','/api/v3/user/login'}
TIMEOUT=12

# A. Create a reproducible local developer project used by the file/code/pytest tools.
(PROJECT/'calculator.py').write_text('''def add(a, b):\n    return a + b\n\ndef subtract(a, b):\n    return a - b\n\ndef multiply(a, b):\n    return a * b\n\ndef divide(a, b):\n    if b == 0:\n        raise ValueError("division by zero")\n    return a / b\n\ndef calculate_square(a):\n    return a * a\n''',encoding='utf-8')
(PROJECT/'app.py').write_text('''from calculator import add, subtract, multiply, divide\n\ndef calculate_demo():\n    return {"sum": add(2, 3), "difference": subtract(5, 2), "product": multiply(3, 4), "quotient": divide(8, 2)}\n\nif __name__ == "__main__":\n    print(calculate_demo())\n''',encoding='utf-8')
(PROJECT/'test_calculator.py').write_text('''from calculator import add, subtract, multiply, divide, calculate_square\nimport pytest\n\ndef test_add(): assert add(2, 3) == 5\ndef test_subtract(): assert subtract(5, 2) == 3\ndef test_multiply(): assert multiply(3, 4) == 12\ndef test_divide(): assert divide(8, 2) == 4\ndef test_divide_by_zero():\n    with pytest.raises(ValueError): divide(1, 0)\ndef test_square(): assert calculate_square(5) == 25\n''',encoding='utf-8')
(PROJECT/'README.md').write_text('''# Sample Developer Project\nA small calculator module used to demonstrate MCP file operations, code search, AST analysis, and pytest automation.\n''',encoding='utf-8')
print('Sample project:',[p.name for p in sorted(PROJECT.iterdir())])

# B. Retrieve a real-world OpenAPI spec. A clear fallback makes the notebook usable during a temporary outage.
try:
    response=requests.get(OPENAPI_URL,timeout=TIMEOUT); response.raise_for_status(); openapi_doc=response.json()
    SPEC_FILE.write_text(json.dumps(openapi_doc,indent=2),encoding='utf-8')
    print('OpenAPI:',openapi_doc.get('openapi'),'| API:',openapi_doc.get('info',{}).get('title'),'| paths:',len(openapi_doc.get('paths',{})))
except Exception as e:
    if SPEC_FILE.exists(): openapi_doc=json.loads(SPEC_FILE.read_text(encoding='utf-8'))
    else: raise RuntimeError(f'Could not retrieve Swagger Petstore OpenAPI spec: {e}')

# C. Client-side policy helpers.
def safe_api_url(path):
    if not isinstance(path,str) or not path.strip(): raise ValueError('Endpoint required')
    url=urljoin(API_BASE+'/',path.lstrip('/')); parsed=urlparse(url)
    if parsed.scheme!='https' or parsed.hostname not in ALLOWED_HOSTS: raise PermissionError('Host outside HTTPS allowlist')
    if parsed.path not in ALLOWED_ENDPOINTS: raise PermissionError('Endpoint not allowlisted: '+parsed.path)
    return url

def safe_project_path(relative_path):
    if not isinstance(relative_path,str) or not relative_path.strip(): raise ValueError('file_path is required')
    candidate=(PROJECT/relative_path).resolve()
    try: candidate.relative_to(PROJECT.resolve())
    except ValueError: raise PermissionError('Path outside project directory')
    if not candidate.exists() or not candidate.is_file(): raise FileNotFoundError('File not found in project: '+relative_path)
    return candidate

# D. Build an actual FastMCP server. All local file tools are confined to PROJECT.
SERVER_SOURCE = r'''import ast, json, subprocess, sys, time
from pathlib import Path
from urllib.parse import urlparse, urljoin
import requests
from mcp.server.fastmcp import FastMCP
mcp=FastMCP("Developer Automation and API Testing MCP Server")
PROJECT=Path("/content/api_testing_agent/sample_developer_project").resolve()
SPEC=Path("/content/api_testing_agent/petstore_openapi.json")
BASE="https://petstore3.swagger.io"
HOSTS={"petstore3.swagger.io"}
PATHS={"/api/v3/pet/findByStatus","/api/v3/pet/1","/api/v3/store/inventory","/api/v3/user/login"}
def safe_project_path(relative_path):
    if not isinstance(relative_path,str) or not relative_path.strip(): raise ValueError("file_path is required")
    candidate=(PROJECT/relative_path).resolve()
    try: candidate.relative_to(PROJECT)
    except ValueError: raise PermissionError("Path outside project directory")
    if not candidate.exists() or not candidate.is_file(): raise FileNotFoundError("File not found in project: "+relative_path)
    return candidate
def safe_url(path):
    if not isinstance(path,str) or not path.strip(): raise ValueError("Endpoint required")
    url=urljoin(BASE+"/",path.lstrip("/")); parsed=urlparse(url)
    if parsed.scheme!="https" or parsed.hostname not in HOSTS: raise PermissionError("Host outside HTTPS allowlist")
    if parsed.path not in PATHS: raise PermissionError("Endpoint not allowlisted: "+parsed.path)
    return url
@mcp.tool()
def list_files() -> dict:
    # List files in the sample project (non-recursive, project-confined).
    return {"status":"success","project":str(PROJECT),"files":sorted(p.name for p in PROJECT.iterdir() if p.is_file())}
@mcp.tool()
def read_file(file_path: str) -> dict:
    # Read a text file from the sample project; traversal is blocked.
    p=safe_project_path(file_path)
    if p.stat().st_size>200000: return {"status":"failed","error":"File exceeds 200 KB limit"}
    return {"status":"success","file":p.name,"content":p.read_text(encoding="utf-8")}
@mcp.tool()
def search_code(query: str, file_pattern: str="*.py") -> dict:
    # Search source-code lines for a literal query within the project.
    if not isinstance(query,str) or not query.strip(): return {"status":"failed","error":"query is required"}
    matches=[]
    for p in sorted(PROJECT.glob(file_pattern)):
        if not p.is_file(): continue
        for n,line in enumerate(p.read_text(encoding="utf-8").splitlines(),1):
            if query.lower() in line.lower(): matches.append({"file":p.name,"line":n,"text":line.strip()})
    return {"status":"success","query":query,"match_count":len(matches),"matches":matches[:100]}
@mcp.tool()
def analyze_python(file_path: str) -> dict:
    # Parse Python source with AST and report functions, classes, imports, and syntax errors.
    p=safe_project_path(file_path)
    if p.suffix!=".py": return {"status":"failed","error":"Only .py files can be analyzed"}
    try: tree=ast.parse(p.read_text(encoding="utf-8"),filename=p.name)
    except SyntaxError as e: return {"status":"failed","syntax_error":str(e),"line":e.lineno}
    functions=[{"name":n.name,"line":n.lineno,"args":[a.arg for a in n.args.args]} for n in ast.walk(tree) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef))]
    classes=[{"name":n.name,"line":n.lineno} for n in ast.walk(tree) if isinstance(n,ast.ClassDef)]
    imports=[]
    for n in ast.walk(tree):
        if isinstance(n,ast.Import): imports.extend(a.name for a in n.names)
        elif isinstance(n,ast.ImportFrom): imports.append(n.module or "")
    return {"status":"success","file":p.name,"function_count":len(functions),"functions":functions,"class_count":len(classes),"classes":classes,"imports":sorted(set(imports))}
@mcp.tool()
def run_tests(test_file: str="test_calculator.py", timeout_seconds: int=10) -> dict:
    # Run pytest on a project test file with a bounded timeout.
    p=safe_project_path(test_file)
    if p.suffix!=".py" or not p.name.startswith("test_"): return {"status":"failed","error":"Only test_*.py files may be passed to pytest"}
    timeout_seconds=max(1,min(int(timeout_seconds),15)); start=time.perf_counter()
    try:
        proc=subprocess.run([sys.executable,"-m","pytest","-q",p.name],cwd=str(PROJECT),capture_output=True,text=True,timeout=timeout_seconds)
        return {"status":"success" if proc.returncode==0 else "failed","return_code":proc.returncode,"elapsed_seconds":round(time.perf_counter()-start,4),"stdout":proc.stdout[-4000:],"stderr":proc.stderr[-2000:]}
    except subprocess.TimeoutExpired:
        return {"status":"error","failure_type":"timeout","retryable":True,"error":f"pytest exceeded {timeout_seconds}s","elapsed_seconds":round(time.perf_counter()-start,4)}
    except Exception as e: return {"status":"error","failure_type":"execution_error","retryable":False,"error":str(e)}
@mcp.tool()
def read_openapi_spec() -> dict:
    # Read summary metadata from the downloaded Petstore OpenAPI document.
    d=json.loads(SPEC.read_text(encoding="utf-8")); return {"status":"success","openapi":d.get("openapi"),"info":d.get("info",{}),"path_count":len(d.get("paths",{})),"schemas":list(d.get("components",{}).get("schemas",{}).keys())[:60]}
@mcp.tool()
def list_api_endpoints() -> dict:
    # List endpoint paths and HTTP methods from the OpenAPI document.
    d=json.loads(SPEC.read_text(encoding="utf-8")); out=[]
    for path,methods in d.get("paths",{}).items():
        for method,meta in methods.items():
            if method.lower() in {"get","post","put","patch","delete","head","options"}: out.append({"path":path,"method":method.upper(),"summary":meta.get("summary","")})
    return {"status":"success","count":len(out),"endpoints":out}
@mcp.tool()
def validate_openapi_spec() -> dict:
    # Validate basic OpenAPI version, paths, path items, and operation shapes.
    d=json.loads(SPEC.read_text(encoding="utf-8")); errors=[]
    if not str(d.get("openapi","")).startswith("3."): errors.append("OpenAPI version is not 3.x")
    if not isinstance(d.get("paths"),dict) or not d["paths"]: errors.append("paths is empty or invalid")
    for path,item in d.get("paths",{}).items():
        if not path.startswith("/"): errors.append("invalid path: "+path)
        if not isinstance(item,dict): errors.append("invalid path item: "+path)
        elif not any(k.lower() in {"get","post","put","patch","delete","head","options"} for k in item): errors.append("no operation defined: "+path)
    return {"status":"success" if not errors else "failed","errors":errors,"path_count":len(d.get("paths",{}))}
@mcp.tool()
def generate_api_test_cases(endpoint: str, method: str="GET") -> dict:
    # Generate safe read-only API test cases for an allowlisted endpoint.
    if method.upper()!="GET": return {"status":"blocked","reason":"GET-only policy"}
    path=urlparse(safe_url(endpoint)).path
    cases=[{"name":"expected_status","params":{},"expected_status":[200]},{"name":"JSON_response","params":{},"expected_status":[200],"expect_json":True},{"name":"response_time","params":{},"expected_status":[200],"max_seconds":12}]
    if path.endswith("findByStatus"): cases.append({"name":"filter_available_pets","params":{"status":"available"},"expected_status":[200]})
    return {"status":"success","endpoint":path,"method":"GET","test_cases":cases}
@mcp.tool()
def execute_api_test(endpoint: str, params: dict=None, expected_status: list=None, method: str="GET") -> dict:
    # Execute an allowlisted HTTPS GET request and validate its response.
    if method.upper()!="GET": return {"status":"blocked","reason":"Only GET is allowed"}
    url=safe_url(endpoint); params=params or {}; expected_status=expected_status or [200]; start=time.perf_counter()
    try:
        r=requests.get(url,params=params,timeout=12,headers={"Accept":"application/json"}); elapsed=round(time.perf_counter()-start,4)
        try: body=r.json(); is_json=True
        except Exception: body=r.text[:250]; is_json=False
        status_ok=r.status_code in expected_status
        format_ok=("json" in r.headers.get("Content-Type","").lower()) if r.status_code<500 else True
        return {"status":"success" if status_ok and format_ok else "failed","endpoint":urlparse(r.url).path,"method":"GET","http_status":r.status_code,"expected_status":expected_status,"status_check_passed":status_ok,"content_type":r.headers.get("Content-Type",""),"json_response":is_json,"format_check_passed":format_ok,"elapsed_seconds":elapsed,"response_preview":body if isinstance(body,(dict,list)) else str(body)[:250]}
    except requests.Timeout: return {"status":"error","failure_type":"timeout","retryable":True,"error":"Request timed out"}
    except Exception as e: return {"status":"error","failure_type":"execution_error","retryable":False,"error":str(e)}
if __name__=="__main__": mcp.run(transport="stdio")
'''
SERVER_FILE.write_text(SERVER_SOURCE,encoding='utf-8')

# E. MCP client, real stdio communication, and dynamic discovery.
MCP_ERRLOG=open(LOG_DIR/'mcp_server_stderr.log','a',buffering=1,encoding='utf-8')
SERVER_PARAMS=StdioServerParameters(command=sys.executable,args=[str(SERVER_FILE)],env={**os.environ,'PYTHONUNBUFFERED':'1'})
def parse_result(r):
    parts=[]
    for c in getattr(r,'content',[]) or []:
        if getattr(c,'text',None):
            try: parts.append(json.loads(c.text))
            except Exception: parts.append(c.text)
    return parts[0] if len(parts)==1 else parts
async def call_tool(session,name,args=None):
    start=time.perf_counter()
    try:
        r=await session.call_tool(name,arguments=args or {}); data=parse_result(r)
        status=data.get('status','success') if isinstance(data,dict) else 'success'
        if getattr(r,'isError',False) and status=='success': status='error'
        return {'tool':name,'status':status,'execution_time':round(time.perf_counter()-start,4),'result':data}
    except Exception as e: return {'tool':name,'status':'error','execution_time':round(time.perf_counter()-start,4),'error':str(e)}
async def discover_tools():
    async with stdio_client(SERVER_PARAMS,errlog=MCP_ERRLOG) as (rs,ws):
        async with ClientSession(rs,ws) as session:
            await session.initialize(); tools=await session.list_tools()
            print('Dynamically discovered MCP tools:')
            for t in tools.tools: print(' -',t.name,'—',t.description)
            return [t.name for t in tools.tools]
available_tools=await discover_tools()

# F. Optional Groq planner. Deterministic fallback is always available.
GROQ_API_KEY=os.environ.get('GROQ_API_KEY','')
if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY=userdata.get('GROQ_API_KEY') or ''
    except Exception: pass
GROQ_MODEL='openai/gpt-oss-20b'; USE_GROQ=bool(GROQ_API_KEY)
def planner_agent(user_request,tool_names):
    if USE_GROQ:
        try:
            from openai import OpenAI
            client=OpenAI(api_key=GROQ_API_KEY,base_url='https://api.groq.com/openai/v1')
            prompt='Return JSON only: {"tasks":[{"step":1,"tool":"...","arguments":{},"depends_on":[]}]} . Available tools='+json.dumps(tool_names)+'. Request='+user_request+'. Do not use write/delete/network methods. Use relative filenames only. Use only GET API tests on allowlisted paths.'
            ans=client.chat.completions.create(model=GROQ_MODEL,temperature=0.1,messages=[{'role':'system','content':'Create a safe, dependency-aware developer automation plan as JSON.'},{'role':'user','content':prompt}])
            obj=json.loads(ans.choices[0].message.content)
            if isinstance(obj.get('tasks'),list): return obj
        except Exception as e: logger.warning('Groq fallback: %s',e)
    return {'tasks':[
      {'step':1,'tool':'list_files','arguments':{},'depends_on':[]},
      {'step':2,'tool':'analyze_python','arguments':{'file_path':'calculator.py'},'depends_on':[]},
      {'step':3,'tool':'search_code','arguments':{'query':'def '},'depends_on':[]},
      {'step':4,'tool':'read_file','arguments':{'file_path':'README.md'},'depends_on':[]},
      {'step':5,'tool':'run_tests','arguments':{'test_file':'test_calculator.py','timeout_seconds':10},'depends_on':[2]},
      {'step':6,'tool':'read_openapi_spec','arguments':{},'depends_on':[]},
      {'step':7,'tool':'validate_openapi_spec','arguments':{},'depends_on':[]},
      {'step':8,'tool':'list_api_endpoints','arguments':{},'depends_on':[6]},
      {'step':9,'tool':'generate_api_test_cases','arguments':{'endpoint':'/api/v3/pet/findByStatus','method':'GET'},'depends_on':[7]},
      {'step':10,'tool':'execute_api_test','arguments':{'endpoint':'/api/v3/store/inventory','params':{},'expected_status':[200],'method':'GET'},'depends_on':[7]}]}

# G. Supervisor and risk-based policy layer.
POLICY={n:{'risk':'LOW','allowed':True} for n in ['list_files','read_file','search_code','analyze_python','read_openapi_spec','list_api_endpoints','validate_openapi_spec','generate_api_test_cases']}
POLICY.update({'run_tests':{'risk':'MEDIUM','allowed':True},'execute_api_test':{'risk':'MEDIUM','allowed':True}})
REQUIRED={'list_files':set(),'read_file':{'file_path'},'search_code':{'query'},'analyze_python':{'file_path'},'run_tests':set(),'read_openapi_spec':set(),'list_api_endpoints':set(),'validate_openapi_spec':set(),'generate_api_test_cases':{'endpoint'},'execute_api_test':{'endpoint'}}
def supervisor_agent(plan,tool_names):
    approved=[]; rejected=[]
    for t in plan.get('tasks',[]):
        name=t.get('tool'); args=t.get('arguments') or {}; reason=None
        if name not in tool_names: reason='Tool not discovered'
        elif name not in POLICY or not POLICY[name]['allowed']: reason='Blocked by policy'
        elif not isinstance(args,dict) or not REQUIRED[name].issubset(args): reason='Required argument missing'
        elif name in {'read_file','analyze_python'}:
            try: safe_project_path(args['file_path'])
            except Exception as e: reason=str(e)
        elif name=='run_tests':
            try:
                safe_project_path(args.get('test_file','test_calculator.py'))
                if not args.get('test_file','test_calculator.py').split('/')[-1].startswith('test_'): reason='Only test_*.py files may be run'
            except Exception as e: reason=str(e)
        elif name=='execute_api_test':
            if args.get('method','GET').upper()!='GET': reason='Only GET is allowed'
            else:
                try: safe_api_url(args['endpoint'])
                except Exception as e: reason=str(e)
        elif name=='generate_api_test_cases':
            if args.get('method','GET').upper()!='GET': reason='Only GET test cases are allowed'
            else:
                try: safe_api_url(args['endpoint'])
                except Exception as e: reason=str(e)
        (rejected if reason else approved).append({**t,**({'rejection_reason':reason} if reason else {})})
    return {'approved':approved,'rejected':rejected}

# H. Persistent JSONL tracing, validation, bounded retry, dependency-aware parallel workflow.
def trace_write(row):
    with open(TRACE_FILE,'a',encoding='utf-8') as f: f.write(json.dumps(row,default=str)+'\n')
async def execute_task(session,t,trace_id,max_retries=1):
    name=t['tool']; args=t.get('arguments',{}); task_id=f'TASK-{int(t["step"]):03d}'
    for attempt in range(max_retries+1):
        start=time.perf_counter(); result=await call_tool(session,name,args); payload=result.get('result')
        status=payload.get('status',result['status']) if isinstance(payload,dict) else result['status']
        if status=='success' and isinstance(payload,dict) and payload.get('status') not in (None,'success'): status=payload['status']
        row={'trace_id':trace_id,'task_id':task_id,'tool':name,'arguments':args,'risk':POLICY.get(name,{}).get('risk'),'attempt':attempt+1,'status':status,'execution_time':round(time.perf_counter()-start,4),'timestamp':datetime.now(timezone.utc).isoformat(),'error':result.get('error') or (payload.get('error') if isinstance(payload,dict) else None),'result':result}
        trace_write(row)
        retryable=isinstance(payload,dict) and payload.get('retryable') is True
        if status=='success' or not retryable or attempt==max_retries:
            return {'step':int(t['step']),'task_id':task_id,'tool':name,'status':status,'execution_time':row['execution_time'],'attempts':attempt+1,'result':payload}
        await asyncio.sleep(0.4)
async def execute_workflow(plan,tool_names):
    trace_id='TRACE-'+uuid.uuid4().hex[:8].upper(); sup=supervisor_agent(plan,tool_names)
    task_map={int(t['step']):t for t in sup['approved']}; done=set(); outputs=[]
    async with stdio_client(SERVER_PARAMS,errlog=MCP_ERRLOG) as (rs,ws):
        async with ClientSession(rs,ws) as session:
            await session.initialize()
            while len(done)<len(task_map):
                ready=[t for step,t in task_map.items() if step not in done and set(t.get('depends_on',[])).issubset(done)]
                if not ready:
                    for step,t in task_map.items():
                        if step not in done: outputs.append({'step':step,'task_id':f'TASK-{step:03d}','tool':t['tool'],'status':'blocked_dependency'}); done.add(step)
                    break
                wave=await asyncio.gather(*[execute_task(session,t,trace_id) for t in ready])
                outputs.extend(wave); done.update(int(t['step']) for t in ready)
                failed={int(r['step']) for r in wave if r['status']!='success'}
                changed=True
                while changed:
                    changed=False
                    for step,t in task_map.items():
                        if step not in done and set(t.get('depends_on',[])) & failed:
                            outputs.append({'step':step,'task_id':f'TASK-{step:03d}','tool':t['tool'],'status':'blocked_dependency'}); done.add(step); failed.add(step); changed=True
    validation=[]
    for x in outputs:
        payload=x.get('result'); valid=x.get('status')=='success'
        if x.get('tool')=='analyze_python' and isinstance(payload,dict): valid=valid and 'function_count' in payload
        if x.get('tool')=='run_tests' and isinstance(payload,dict): valid=valid and payload.get('return_code')==0
        if x.get('tool')=='execute_api_test' and isinstance(payload,dict): valid=valid and payload.get('status')=='success' and payload.get('status_check_passed') is True and payload.get('format_check_passed') is True
        validation.append({'task_id':x.get('task_id'),'tool':x.get('tool'),'valid':valid})
    return {'trace_id':trace_id,'supervision':sup,'execution':outputs,'validation':{'overall_valid':all(v['valid'] for v in validation) and not sup['rejected'],'items':validation}}

USER_REQUEST='List project files, analyze calculator.py, search for function definitions, read README.md, run pytest after analysis, inspect and validate Petstore OpenAPI, generate API cases, and run an approved read-only API check.'
plan=planner_agent(USER_REQUEST,available_tools); print('\nPlanner:', 'Groq' if USE_GROQ else 'deterministic fallback'); print('PLAN:',json.dumps(plan,indent=2))
workflow=await execute_workflow(plan,available_tools)
print('\nSUPERVISOR REJECTIONS:',json.dumps(workflow['supervision']['rejected'],indent=2))
print('\nWORKFLOW RESULTS:')
for item in workflow['execution']:
    print(item.get('task_id'),item.get('tool'),'=>',item.get('status'),'in',item.get('execution_time','-'),'seconds')
print('\nVALIDATION:',json.dumps(workflow['validation'],indent=2))

# I. Required performance variation: same four developer-tool calls, sequential vs parallel, five runs.
async def benchmark(tasks,repeats=5):
    seq=[]; par=[]
    for _ in range(repeats):
        async with stdio_client(SERVER_PARAMS,errlog=MCP_ERRLOG) as (rs,ws):
            async with ClientSession(rs,ws) as session:
                await session.initialize(); start=time.perf_counter()
                for t in tasks: await call_tool(session,t['tool'],t.get('arguments',{}))
                seq.append(time.perf_counter()-start)
        async with stdio_client(SERVER_PARAMS,errlog=MCP_ERRLOG) as (rs,ws):
            async with ClientSession(rs,ws) as session:
                await session.initialize(); start=time.perf_counter()
                await asyncio.gather(*[call_tool(session,t['tool'],t.get('arguments',{})) for t in tasks])
                par.append(time.perf_counter()-start)
    a,b=mean(seq),mean(par)
    return {'task_count':len(tasks),'repeats':repeats,'sequential_seconds':[round(x,4) for x in seq],'parallel_seconds':[round(x,4) for x in par],'average_sequential_seconds':round(a,4),'average_parallel_seconds':round(b,4),'relative_time_reduction_percent':round((a-b)/a*100,2) if a else 0}
bench_tasks=[{'tool':'analyze_python','arguments':{'file_path':'calculator.py'}},{'tool':'search_code','arguments':{'query':'def '}},{'tool':'list_files','arguments':{}},{'tool':'read_file','arguments':{'file_path':'README.md'}}]
performance=await benchmark(bench_tasks,5)
print('\nPERFORMANCE COMPARISON (actual measured values):',json.dumps(performance,indent=2))

# J. Demonstrate input, security, and policy rejection (nothing unsafe is executed).
bad_plan={'tasks':[
 {'step':1,'tool':'read_file','arguments':{'file_path':'../../etc/passwd'},'depends_on':[]},
 {'step':2,'tool':'execute_api_test','arguments':{'endpoint':'https://example.com/','method':'GET'},'depends_on':[]},
 {'step':3,'tool':'execute_api_test','arguments':{'endpoint':'/api/v3/pet/1','method':'POST'},'depends_on':[]},
 {'step':4,'tool':'unknown_tool','arguments':{},'depends_on':[]}]}
rejection_demo=supervisor_agent(bad_plan,available_tools)
print('\nEXPECTED SECURITY/POLICY REJECTIONS:',json.dumps(rejection_demo,indent=2))

# K. Structured report and persistent artifacts.
records=workflow['execution']; success=sum(x.get('status')=='success' for x in records); failed=sum(x.get('status') not in {'success','blocked_dependency'} for x in records)
report={'experiment':'Experiment 06','title':'Developer Automation and API Testing Agent using MCP Protocol','generated_at':datetime.now(timezone.utc).isoformat(),'user_request':USER_REQUEST,'llm':{'provider':'Groq' if USE_GROQ else 'deterministic fallback','model':GROQ_MODEL if USE_GROQ else None},'mcp_tools':available_tools,'planning':plan,'supervision':workflow['supervision'],'execution':records,'validation':workflow['validation'],'policy_rejection_demo':rejection_demo,'observability':{'trace_id':workflow['trace_id'],'trace_file':str(TRACE_FILE),'log_file':str(LOG_FILE)},'performance':performance,'dop_dos':{'parallel_phase':'list_files, analyze_python, search_code, read_file and independent OpenAPI checks','dop_observed_maximum':4,'sequential_dependency':'run_tests depends on successful completion of analyze_python','model':'hybrid parallel and sequential execution'},'summary':{'planned_tasks':len(plan.get('tasks',[])),'executed_records':len(records),'successful_tasks':success,'failed_tasks':failed,'rejected_tasks':len(workflow['supervision']['rejected']),'overall_valid':workflow['validation']['overall_valid']},'artifacts':{'log':str(LOG_FILE),'trace':str(TRACE_FILE),'report':str(REPORT_FILE),'summary':str(SUMMARY_FILE),'server_stderr':str(LOG_DIR/'mcp_server_stderr.log')}}
REPORT_FILE.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8'); SUMMARY_FILE.write_text(json.dumps(report['summary'],indent=2),encoding='utf-8')
print('\nFINAL REPORT SUMMARY:',json.dumps(report['summary'],indent=2))
print('\nARTIFACTS:')
for path in [LOG_FILE,TRACE_FILE,REPORT_FILE,SUMMARY_FILE,LOG_DIR/'mcp_server_stderr.log']:
    print(path,'exists=',path.exists(),'bytes=',path.stat().st_size if path.exists() else 0)
print('\nLearning outcomes: MCP server/client, dynamic discovery, file tools, code search, AST analysis, pytest automation, API testing, planning/supervision, policy validation, dependencies, recovery, JSONL tracing, performance measurement, structured reports.')
print('Conclusion: The combined system exposes developer and API testing tools through MCP, with controlled execution, validation, tracing, and performance monitoring.')
try:
    from google.colab import files
    for path in [REPORT_FILE,SUMMARY_FILE,TRACE_FILE]:
        if path.exists(): files.download(str(path))
except Exception: pass


Sample project: ['README.md', 'app.py', 'calculator.py', 'test_calculator.py']
OpenAPI: 3.0.4 | API: Swagger Petstore - OpenAPI 3.0 | paths: 13
Dynamically discovered MCP tools:
 - list_files — 
 - read_file — 
 - search_code — 
 - analyze_python — 
 - run_tests — 
 - read_openapi_spec — 
 - list_api_endpoints — 
 - validate_openapi_spec — 
 - generate_api_test_cases — 
 - execute_api_test — 

Planner: Groq
PLAN: {
  "tasks": [
    {
      "step": 1,
      "tool": "list_files",
      "arguments": {},
      "depends_on": []
    },
    {
      "step": 2,
      "tool": "read_file",
      "arguments": {
        "path": "calculator.py"
      },
      "depends_on": [
        1
      ]
    },
    {
      "step": 3,
      "tool": "analyze_python",
      "arguments": {
        "path": "calculator.py"
      },
      "depends_on": [
        2
      ]
    },
    {
      "step": 4,
      "tool": "search_code",
      "arguments": {
        "path": "calculator.py",
        "pattern": "def "
      },


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>